# Testing the new flow

In [3]:
import sys
sys.path.insert(1, '..')
from ps_features_builder import PSFeaturesBuilder
from effect_estimation import EffectEstimation
from ps_matching import PSMatching
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os 

PATH_DATA = '../../data/'
PATHS = {
    'vialidades': os.path.join(PATH_DATA, 'vialidades.json'),
    'speed_cameras': os.path.join(PATH_DATA, 'fotocivicas-ubicacion-puntos', 'fotocivicas-ubicacion-puntos.shp'),
    'metro_coordinates': os.path.join(PATH_DATA, 'metro-station-coordinates.parquet'),
    'afluencia_metro': os.path.join(PATH_DATA, 'afluencia-metro-semanal.parquet'),
    'classified_incidents': os.path.join(PATH_DATA, 'classified-incidents.parquet'),
    'volumen_mensual': os.path.join(PATH_DATA, 'volumen-total-mensual.parquet')
}

resumen_efectos = pd.read_parquet(os.path.join(PATH_DATA, "resumen-efectos-con-offsets.parquet"))

In [2]:
tests = {
    123: {
        'base': {'x': 0, 'y': 0},
        'mean_dtc': {'x': 130, 'y': 80},
        'median_dtc': {'x': 140, 'y': 110},
        'random': {'x': 30, 'y': 80},
        'std_dtc': {'x': 0, 'y': 80}
    },
    185: {
        'base': {'x': 0, 'y': 0},
        'mean_dtc': {'x': 0, 'y': 60},
        'median_dtc': {'x': 0, 'y': 50},
        'random': {'x': 0, 'y': 10},
        'std_dtc': {'x': 80, 'y': 20}
    },
    368: {
        'base': {'x': 0, 'y': 0},
        'mean_dtc': {'x': 30, 'y': 0},
        'median_dtc': {'x': 20, 'y': 0},
        'random': {'x': 40, 'y': 20},
        'std_dtc': {'x': 40, 'y': 40}
    },
    500: {
        'base': {'x': 0, 'y': 0},
        'mean_dtc': {'x': 20, 'y': 0},
        'median_dtc': {'x': 20, 'y': 0},
        'random': {'x': 10, 'y': 20},
        'std_dtc': {'x': 30, 'y': 20}
    }
}

In [3]:
efectos = []

for grid_size, grid_tests in tests.items():
    break
    for case, offsets in grid_tests.items():
        print(grid_size, case, offsets, end='\r')
        
        # Create grid and features
        ps_builder = PSFeaturesBuilder(
            PATHS, 
            grid_size=grid_size, 
            offset_x_m=offsets.get('x'), 
            offset_y_m=offsets.get('y')
        )
        ps_builder.build()
        
        # Propensity score matching
        ps_matcher = PSMatching(
            ps_features=ps_builder.ps_features,
            grid=ps_builder.grid,
            outcome=ps_builder.outcome,
            grid_size=grid_size
        )
        ps_matcher.build()
        
        # Estimate effects
        effect_estimator = EffectEstimation(
            outcome=ps_matcher.outcome,
            matched_grids=ps_matcher.matched_grids,
            inicio_operaciones=ps_builder.inicio_operaciones
        )
        effect_estimator.estimate_all()
        
        # Guardamos el resumen
        resumen_efectos = (
            effect_estimator
            .get_all_treatment_effects()
            .assign(
                grid_size=grid_size,
                offset_x=ps_builder.offset_x_m,
                offset_y=ps_builder.offset_y_m,
                case=case
            )
        )
        efectos.append(resumen_efectos)

In [232]:
def clean_ax(ax):
    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)
    ax.spines.left.set_visible(False)

    axis_color = "#6c757d"
    ax.xaxis.label.set_color(axis_color)
    ax.yaxis.label.set_color(axis_color)
    ax.spines.bottom.set_color("white")
    ax.spines.bottom.set_visible(False)
    ax.spines.left.set_color(axis_color)
    ax.tick_params(axis='both', colors=axis_color)
    ax.grid(axis='y', alpha=.2, linestyle=':')
    
    ttl = ax.title
    ttl.set_position([.5, 2])

In [350]:
outcome_type = "tasas"
grid_sizes = list(tests.keys())   
variables = ["total", "min", "pic", "fcs"]  

markers = {
    "base": "|",
    "random": "o",
    "mean_dtc": "s",
    "median_dtc": "p",
    "std_dtc": "^"
}

grid_size_meters = {
    123: "0 - 300 m",
    185: "0 - 200 m",
    368: "0 - 100 m",
    500: "0 - 75 m"
}

n_rows = len(grid_sizes)
n_cols = len(variables)

# ---- FIGURA Y GRIDSPEC ----
fig = plt.figure(figsize=(10, 4))

gs = fig.add_gridspec(
    n_rows + 1,
    n_cols,
    height_ratios=[1]*n_rows + [1]   # espacio pequeño para la leyenda
)

# matriz de ejes normales
axes = np.array([
    [fig.add_subplot(gs[i, j]) for j in range(n_cols)]
    for i in range(n_rows)
])

# eje único para la leyenda
legend_ax = fig.add_subplot(gs[-1, :])
legend_ax.spines.top.set_color('darkgray')
legend_ax.spines.right.set_visible(False)
legend_ax.spines.left.set_visible(False)
legend_ax.spines.bottom.set_visible(False)
legend_ax.set_xticks([])
legend_ax.set_yticks([])

legend_handles = []

# ---- LÍMITES POR VARIABLE ----
limits_by_var = {}
for var in variables:
    coefs_var = resumen_efectos.query(
        "outcome_type == @outcome_type and variable == @var"
    ).coeficiente.values
    limits_by_var[var] = (
        coefs_var.min() * 1.3 if var != 'pic' else coefs_var.min()*4,
        coefs_var.max() * 1.1 if var != 'fcs' else coefs_var.max()*4
    )

# ---- LOOP PRINCIPAL ----
for i, grid_size in enumerate(grid_sizes):
    for j, variable in enumerate(variables):

        ax = axes[i, j]
        clean_ax(ax)

        df = resumen_efectos.query(
            "outcome_type == @outcome_type and grid_size == @grid_size and variable == @variable"
        )

        yname = f"{grid_size_meters.get(grid_size)}"

        # ---- SCATTER ----
        for case, marker in markers.items():
            row = df[df.case == case].iloc[0]

            sca = ax.scatter(
                x=[row.coeficiente],
                y=[yname],
                marker=marker,
                color="black" if row.valor_p > 0.1 else "red",
                s=70 if case == "base" else None,
                label=case if (i == 0 and j == 0) else None
            )

            if i == 0 and j == 0:
                legend_handles.append(sca)

        # ---- LÍNEA HORIZONTAL ----
        ax.axhline(
            y=yname,
            xmin=0.05, xmax=0.95,
            color="lightgray",
            linestyle=":",
            zorder=-1
        )

        # ---- XTICKS internos ----
        coef_base = df[df.case == "base"].coeficiente.values[0]
        xticks = {df.coeficiente.min(), df.coeficiente.max(), coef_base}
        ax.set_xticks(list(xticks))

        # ---- LÍMITES POR VARIABLE ----
        xmin, xmax = limits_by_var[variable]
        ax.set_xlim(xmin, xmax)

        # ---- TITLES EN LA FILA SUPERIOR ----
        if i == 0:
            ax.set_title(variable, fontsize=9, color="gray", ha='center')

        # ---- LABELS EJE Y SOLO IZQUIERDA ----
        if j == 0:
            ax.set_yticks([yname])
            ax.set_yticklabels([yname])
        else:
            ax.set_yticks([])
            ax.set_yticklabels([])

        # ---- OCULTAR XTICKS excepto última fila ----
        if i < n_rows - 1:
            ax.set_xticks([])
            ax.set_xticklabels([])
        else:
            ax.set_xticks([xmin, xmax])
            ax.set_xticklabels([f"{xmin:.2e}", f"{xmax:.2e}"], ha='center')

# ---- LEYENDA ----
legend_ax.legend(
    handles=legend_handles,
    labels=markers.keys(),
    loc="center",
    ncols=5,
    fontsize=9,
    frameon=False
)

fig.suptitle(
    f"Coeficientes y signifancia en {outcome_type} de accidentes",
    ha="center",
    color="gray",
    #x=0.015
)

fig.tight_layout()
#fig.savefig(os.path.join(PATH_DATA, "graphs", "coefficients-on-tasas-accidents.png"), dpi=300, transparent=True)
plt.close()